# DC Motor (Powertrain) Scenario Sample Generator & Visualiser

Generates synthetic brushless DC motor sensor time series for **load-testing and benchmarking ML infrastructure** — not for model training.

Data is generated using NASA's [progpy](https://nasa.github.io/progpy/) **Powertrain** model (ESC + DCMotor + PropellerLoad) with configurable degradation, decoy events, and planned maintenance windows.

Since the Powertrain model has **no built-in failure events**, degradation is modelled externally by evolving motor parameters over time. A calibration grid of short progpy simulations maps parameter values to steady-state signals.

Runs all **10 motor scenarios** (1 normal + 3 failures + 6 decoys):

| Group | Failure mode | Physics | Hard negatives |
|-------|-------------|---------|----------------|
| Winding | R ↑ (0.081 → 0.30 Ω) | Speed drops, current rises (I²R loss) | Load step/ramp (speed ↓, but current tracks proportionally) |
| Bearing | B ↑ (0 → 0.0015) | Friction rises, speed drops, power lost to heat | Voltage sag step/ramp (speed ↓, but no friction) |
| Demag | K ↓ (0.0265 → 0.012) | Back-emf drops, torque-current coupling shifts | Duty cycle step/ramp (speed ↓, but consistent K) |

In [ ]:
duration_days = 365
save_freq_s = 60
decoy_freq_per_day = 2.0
maintenance_freq_per_month = 1.0
seed = 42
run_short_scenarios = True
run_long_series = True
downsample = 50
failure_severity = 0
ambient_var_K = 5.0
duty_cycle_var = 0.5

In [ ]:
import sys, importlib, warnings
warnings.filterwarnings('ignore')

# Make dcmotor_generator importable and reload if already loaded
if "dcmotor_generator" in sys.modules:
    importlib.reload(sys.modules["dcmotor_generator"])
from dcmotor_generator import build_scenarios, run_all

## Run all 10 simulations
First builds a calibration grid (~90 short progpy Powertrain sims at dt=2e-5), then generates 6h scenarios by interpolating the grid. Each scenario produces ~361 rows (6h at 1 sample/min).

In [ ]:
import os, glob

if run_short_scenarios:
    # Check if short scenario CSVs already exist
    existing = glob.glob("motor_sample_data/normal.csv")
    if existing and os.path.exists("motor_sample_data/scenarios.csv"):
        print("Loading existing short scenario data from motor_sample_data/*.csv")
        import pandas as pd
        scenarios_meta = pd.read_csv("motor_sample_data/scenarios.csv")
        results = {}
        for _, row in scenarios_meta.iterrows():
            path = f"motor_sample_data/{row['scenario']}.csv"
            if os.path.exists(path):
                results[row["scenario"]] = pd.read_csv(path)
        print(f"  Loaded {len(results)} scenarios")
    else:
        print("Generating short scenarios...")
        scenarios = build_scenarios()
        results = run_all(scenarios)
else:
    results = None
    print("Skipped short scenarios (run_short_scenarios=False)")

In [ ]:
if results:
    import pandas as pd
    rows = []
    for name, df in results.items():
        vc = df["label"].value_counts()
        rows.append({
            "scenario": name, "rows": len(df),
            "NORMAL": vc.get("NORMAL", 0),
            "PRE_FAILURE": vc.get("PRE_FAILURE", 0),
            "FAILURE": vc.get("FAILURE", 0),
        })
    display(pd.DataFrame(rows).set_index("scenario"))

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Image
from dcmotor_generator import COLOURS, LABEL_ALPHA, LABEL_COLOUR

def show_fig(fig):
    """Show interactive plotly + static PNG fallback (for GitHub rendering)."""
    fig.show()
    try:
        display(Image(fig.to_image(format="png", width=1400, height=500, scale=2)))
    except Exception:
        pass  # kaleido may not be available in headless mode

## Plot 0 — Healthy baseline
All signals stable, every row labelled NORMAL. This is what "nothing is wrong" looks like for the motor.

In [ ]:
if results:
    df = results["normal"]
    pairs = [
        ("rotational_velocity_rads", "Rotational velocity (rad/s)"),
        ("current_rms_A",           "RMS current (A)"),
        ("torque_load_Nm",          "Torque load (Nm)"),
        ("mechanical_power_W",      "Mechanical power (W)"),
        ("electrical_power_W",      "Electrical power (W)"),
        ("resistance_ohm",          "Winding resistance (Ω)"),
    ]
    fig = make_subplots(rows=2, cols=3, subplot_titles=[p[1] for p in pairs])
    for idx, (col, label) in enumerate(pairs):
        r, c = idx // 3 + 1, idx % 3 + 1
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df["time_h"], y=df[col], mode="lines",
                                     line=dict(color=COLOURS["normal"], width=1.5),
                                     name=label, showlegend=False), row=r, col=c)
            fig.update_xaxes(title_text="Time (hours)", row=r, col=c)
    fig.update_layout(title="Healthy motor — baseline signals (all NORMAL)",
                      height=500, template="plotly_white")
    show_fig(fig)

## Plot 1 — Winding degradation (R↑) vs load-change decoys

**Both** show speed dropping — but the failure also shows **current rising disproportionately**.

- **Failure:** R increases → I²R losses grow → current rises faster than speed drops
- **Hard negative:** Higher propeller load → speed ↓ and current ↑, but proportionally — no excess I²R loss

The discrimination problem: you can't just threshold on "speed is dropping" — you need to check whether the current rise is explained by the load change.

In [ ]:
LABEL_COLORS_RGBA = {
    "NORMAL":      "rgba(29,158,117,0.06)",
    "PRE_FAILURE": "rgba(186,117,23,0.18)",
    "FAILURE":     "rgba(216,90,48,0.25)",
}

def _add_label_shading(fig, df, time_col="time_h"):
    """Add background shading for NORMAL / PRE_FAILURE / FAILURE labels."""
    labels = df["label"].values
    time = df[time_col].values
    prev_lbl = labels[0]
    start = time[0]
    for i in range(1, len(labels)):
        if labels[i] != prev_lbl or i == len(labels) - 1:
            end = time[i]
            color = LABEL_COLORS_RGBA.get(prev_lbl)
            if color and prev_lbl != "NORMAL":
                fig.add_vrect(x0=start, x1=end, fillcolor=color,
                              layer="below", line_width=0)
            prev_lbl = labels[i]
            start = time[i]

def plot_group_interactive(scenarios, signal_y, signal_x, title, y_label, x_label, scatter_label):
    """3-panel interactive plot: signal_y over time, signal_x over time, scatter x vs y.
    Failure scenarios get PRE_FAILURE (orange) and FAILURE (red) background shading."""
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=[f"{y_label} over time",
                                        f"{x_label} over time",
                                        f"{scatter_label}"])
    for name, label, colour in scenarios:
        df = results[name]
        fig.add_trace(go.Scatter(x=df["time_h"], y=df[signal_y], mode="lines",
                                 line=dict(color=colour, width=1.5),
                                 name=label, legendgroup=label), row=1, col=1)
        fig.add_trace(go.Scatter(x=df["time_h"], y=df[signal_x], mode="lines",
                                 line=dict(color=colour, width=1.5),
                                 name=label, legendgroup=label, showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=df[signal_x], y=df[signal_y], mode="markers",
                                 marker=dict(color=colour, size=3, opacity=0.5),
                                 name=label, legendgroup=label, showlegend=False), row=1, col=3)

        # Add PRE_FAILURE / FAILURE shading on time panels for faulty scenarios
        if "label" in df.columns and df["label"].nunique() > 1:
            for lbl, rgba in LABEL_COLORS_RGBA.items():
                if lbl == "NORMAL":
                    continue
                mask = df["label"] == lbl
                if not mask.any():
                    continue
                changes = mask.astype(int).diff().fillna(0)
                starts = df["time_h"][changes == 1].values
                ends = df["time_h"][changes == -1].values
                if mask.iloc[-1]:
                    ends = np.append(ends, df["time_h"].iloc[-1])
                for s, e in zip(starts, ends):
                    fig.add_vrect(x0=s, x1=e, fillcolor=rgba, layer="below",
                                  line_width=0, row=1, col=1)
                    fig.add_vrect(x0=s, x1=e, fillcolor=rgba, layer="below",
                                  line_width=0, row=1, col=2)

    fig.update_xaxes(title_text="Time (hours)", row=1, col=1)
    fig.update_xaxes(title_text="Time (hours)", row=1, col=2)
    fig.update_xaxes(title_text=x_label, row=1, col=3)
    fig.update_yaxes(title_text=y_label, row=1, col=1)
    fig.update_yaxes(title_text=x_label, row=1, col=2)
    fig.update_yaxes(title_text=y_label, row=1, col=3)
    fig.update_layout(title=title, height=450, template="plotly_white")
    return fig

if results:
    fig = plot_group_interactive(
        scenarios=[
            ("normal",                    "Healthy",       COLOURS["normal"]),
            ("motor_winding_degradation", "Winding degr.", COLOURS["failure"]),
            ("decoy_load_step",           "Load step",     COLOURS["decoy"]),
            ("decoy_load_ramp",           "Load ramp",     "#2196F3"),
        ],
        signal_y="current_rms_A", signal_x="rotational_velocity_rads",
        title="Winding degradation (R↑) vs load-change decoys<br><sub>Failure: current HIGHER than expected for speed (I²R loss) | Decoy: current tracks speed proportionally</sub>",
        y_label="RMS current (A)", x_label="Rotational velocity (rad/s)",
        scatter_label="Speed vs Current (discriminator)")
    show_fig(fig)

## Plot 2 — Bearing wear (B↑) vs voltage-sag decoys

**Both** show speed dropping — but the failure shows **mechanical power lower than expected** for the speed.

- **Failure:** Friction absorbs energy → less power reaches the propeller at the same speed
- **Hard negative:** Lower voltage → speed drops, but power-speed relationship stays consistent

You can't just watch speed. You need to check: "is the power drop explained by the voltage, or is there unexplained energy loss?"

In [ ]:
if results:
    fig = plot_group_interactive(
        scenarios=[
            ("normal",                 "Healthy",          COLOURS["normal"]),
            ("motor_bearing_wear",     "Bearing wear",     COLOURS["failure"]),
            ("decoy_voltage_sag_step", "Voltage sag step", COLOURS["decoy"]),
            ("decoy_voltage_sag_ramp", "Voltage sag ramp", "#2196F3"),
        ],
        signal_y="mechanical_power_W", signal_x="rotational_velocity_rads",
        title="Bearing wear (B↑) vs voltage-sag decoys<br><sub>Failure: power LOWER than expected for speed (friction absorbs energy) | Decoy: power tracks speed proportionally</sub>",
        y_label="Mechanical power (W)", x_label="Rotational velocity (rad/s)",
        scatter_label="Speed vs Mech Power (discriminator)")
    show_fig(fig)

## Plot 3 — Demagnetization (K↓) vs duty-cycle decoys

**Both** show speed changing — but the failure shifts the **electrical power vs torque** relationship.

- **Failure:** K drops → torque constant changes → different power-torque coupling
- **Hard negative:** Reduced duty cycle → less voltage applied → speed/power drop but K stays constant

The discrimination problem: speed drops in both cases, but the underlying torque-current relationship changes only in the failure case.

In [ ]:
if results:
    fig = plot_group_interactive(
        scenarios=[
            ("normal",                "Healthy",         COLOURS["normal"]),
            ("motor_demagnetization", "Demagnetization", COLOURS["failure"]),
            ("decoy_duty_step",       "Duty step",       COLOURS["decoy"]),
            ("decoy_duty_ramp",       "Duty ramp",       "#2196F3"),
        ],
        signal_y="electrical_power_W", signal_x="torque_load_Nm",
        title="Demagnetization (K↓) vs duty-cycle decoys<br><sub>Failure: elec power shifts (K affects torque-current coupling) | Decoy: power scales with duty (consistent K)</sub>",
        y_label="Electrical power (W)", x_label="Torque load (Nm)",
        scatter_label="Torque vs Elec Power (discriminator)")
    show_fig(fig)

---
# Long Time Series (365 days)

Generate 1-year data with decoy events mixed into every series (~2/day). Each series saved as parquet. Uses the same tiling strategy as the pump generator: build a healthy template, tile it for the full duration, then splice in degradation episodes and decoy events.

In [ ]:
if run_long_series:
    import pandas as pd, json
    from pathlib import Path

    long_names = ["normal_long", "winding_failure_long",
                  "bearing_failure_long", "demag_failure_long"]

    # Check if cached data matches current parameters
    config_path = Path("motor_sample_data/long_series_config.json")
    current_config = dict(
        duration_days=duration_days, save_freq_s=save_freq_s,
        decoy_freq_per_day=decoy_freq_per_day, failure_severity=failure_severity,
        maintenance_freq_per_month=maintenance_freq_per_month,
        ambient_var_K=ambient_var_K, duty_cycle_var=duty_cycle_var, seed=seed,
    )

    cache_valid = all(Path(f"motor_sample_data/{n}.parquet").exists() for n in long_names)
    if cache_valid and config_path.exists():
        cached_config = json.loads(config_path.read_text())
        if cached_config != current_config:
            print(f"Parameters changed — regenerating (was {cached_config.get('duration_days')}d, "
                  f"now {duration_days}d)")
            cache_valid = False

    if cache_valid:
        print("Loading existing long series from motor_sample_data/*.parquet")
        long_results = {}
        for n in long_names:
            long_results[n] = pd.read_parquet(f"motor_sample_data/{n}.parquet")
            print(f"  {n}: {len(long_results[n])} rows")
    else:
        print("Generating long series...")
        from dcmotor_generator import generate_dataset
        long_results = generate_dataset(
            duration_days=duration_days,
            save_freq_s=save_freq_s,
            decoy_freq_per_day=decoy_freq_per_day,
            failure_severity=failure_severity,
            maintenance_freq_per_month=maintenance_freq_per_month,
            ambient_var_K=ambient_var_K,
            duty_cycle_var=duty_cycle_var,
            seed=seed,
        )
        config_path.write_text(json.dumps(current_config, indent=2))
else:
    long_results = None
    print("Skipped long series (run_long_series=False)")

In [ ]:
if long_results:
    rows = []
    for name, df in long_results.items():
        vc = df["label"].value_counts()
        n_decoys = int((df["event_type"].str.startswith("decoy_")).sum())
        rows.append({
            "series": name, "rows": len(df),
            "days": f'{len(df)*save_freq_s/86400:.0f}',
            "NORMAL": vc.get("NORMAL", 0),
            "PRE_FAILURE": vc.get("PRE_FAILURE", 0),
            "FAILURE": vc.get("FAILURE", 0),
            "decoy_rows": n_decoys,
        })
    display(pd.DataFrame(rows).set_index("series"))

In [ ]:
def plot_long_series(df, title, signals=None, ds_factor=downsample):
    """Plot a full year-long series with event shading. Downsamples for performance."""
    if signals is None:
        signals = ["rotational_velocity_rads", "current_rms_A",
                    "mechanical_power_W", "electrical_power_W", "torque_load_Nm"]

    ds = df.iloc[::ds_factor]
    time_days = ds["time_h"] / 24.0

    # Color map for event types
    event_colors = {
        "normal": "rgba(0,0,0,0)",
        "maintenance": "rgba(100,100,100,0.15)",
        "startup": "rgba(100,100,100,0.08)",
    }
    for c in ds["event_type"].unique():
        if c.startswith("failure_"):
            event_colors[c] = "rgba(216,90,48,0.25)"
        elif c.startswith("decoy_"):
            event_colors[c] = "rgba(29,158,117,0.15)"
        elif c == "post_failure":
            event_colors[c] = "rgba(216,90,48,0.12)"

    fig = make_subplots(
        rows=len(signals), cols=1, shared_xaxes=True,
        subplot_titles=signals, vertical_spacing=0.03,
    )

    for i, sig in enumerate(signals):
        row = i + 1
        fig.add_trace(go.Scattergl(
            x=time_days, y=ds[sig], mode="lines",
            line=dict(width=0.5, color="#333"),
            name=sig, showlegend=False,
        ), row=row, col=1)

        if i == 0:
            for evt in ds["event_type"].unique():
                if evt == "normal":
                    continue
                mask = ds["event_type"] == evt
                if not mask.any():
                    continue
                color = event_colors.get(evt, "rgba(150,150,150,0.15)")
                changes = mask.astype(int).diff().fillna(0)
                starts = time_days[changes == 1].values
                ends = time_days[changes == -1].values
                if mask.iloc[-1]:
                    ends = np.append(ends, time_days.iloc[-1])
                for s, e in zip(starts, ends):
                    fig.add_vrect(x0=s, x1=e, fillcolor=color, layer="below",
                                  line_width=0)

    fig.update_xaxes(title_text="Time (days)", row=len(signals), col=1)
    fig.update_layout(
        title=title,
        height=200 * len(signals),
        template="plotly_white",
        showlegend=False,
    )
    return fig

### Healthy series (365 days, ~730 decoy events, no failure)
Green shading = decoy events. Zoom in with plotly to inspect individual events.

In [ ]:
if long_results:
    fig = plot_long_series(long_results["normal_long"],
                           f"Normal (healthy) motor — {duration_days:.0f} days")
    show_fig(fig)

### Winding failure series (365 days)
Red shading = failure episode. Green = decoy events throughout.

In [ ]:
if long_results:
    fig = plot_long_series(long_results["winding_failure_long"],
                           f"Winding failure (R↑) — {duration_days:.0f} days")
    show_fig(fig)

### Bearing failure series (365 days)

In [ ]:
if long_results:
    fig = plot_long_series(long_results["bearing_failure_long"],
                           f"Bearing failure (B↑) — {duration_days:.0f} days")
    show_fig(fig)

### Demagnetization failure series (365 days)

In [ ]:
if long_results:
    fig = plot_long_series(long_results["demag_failure_long"],
                           f"Demagnetization failure (K↓) — {duration_days:.0f} days")
    show_fig(fig)

### Zoom: failure windows at full resolution
Shows 24h around each failure event — the transition from normal → pre-failure → failure at 1-min resolution.

In [ ]:
def plot_failure_zoom(df, series_name, window_hours=24):
    """Zoom into the failure window at full resolution."""
    fail_idx = df.index[df["label"] == "FAILURE"]
    if len(fail_idx) == 0:
        print(f"  No failure in {series_name}")
        return None
    fail_row = fail_idx[0]
    rows_per_h = 3600 // save_freq_s
    half_window = int(window_hours / 2 * rows_per_h)
    start = max(0, fail_row - half_window)
    end = min(len(df), fail_row + half_window)
    window = df.iloc[start:end]
    signals = ["rotational_velocity_rads", "current_rms_A",
               "mechanical_power_W", "electrical_power_W", "torque_load_Nm"]
    time_h = window["time_h"]
    fig = make_subplots(rows=len(signals), cols=1, shared_xaxes=True,
                        subplot_titles=signals, vertical_spacing=0.04)
    label_colors = {"NORMAL": "#1D9E75", "PRE_FAILURE": "#BA7517", "FAILURE": "#D85A30"}
    for i, sig in enumerate(signals):
        row = i + 1
        for lbl, color in label_colors.items():
            mask = window["label"] == lbl
            if mask.any():
                fig.add_trace(go.Scatter(
                    x=time_h[mask], y=window[sig][mask], mode="markers+lines",
                    marker=dict(size=2, color=color), line=dict(width=1, color=color),
                    name=lbl if i == 0 else None, showlegend=(i == 0), legendgroup=lbl,
                ), row=row, col=1)
        for evt in window["event_type"].unique():
            if evt == "normal":
                continue
            mask = window["event_type"] == evt
            changes = mask.astype(int).diff().fillna(0)
            starts = time_h[changes == 1].values
            ends = time_h[changes == -1].values
            if mask.iloc[-1]:
                ends = np.append(ends, time_h.iloc[-1])
            fc = "rgba(216,90,48,0.12)" if "failure" in evt else "rgba(29,158,117,0.12)"
            for s, e in zip(starts, ends):
                fig.add_vrect(x0=s, x1=e, fillcolor=fc, line_width=0, row=row, col=1)
    fail_day = df.iloc[fail_row]["time_h"] / 24
    fig.update_xaxes(title_text="Time (hours)", row=len(signals), col=1)
    fig.update_layout(
        title=f"{series_name} — {window_hours}h around failure (day {fail_day:.1f})",
        height=180 * len(signals), template="plotly_white")
    return fig

if long_results:
    for name in ["winding_failure_long", "bearing_failure_long", "demag_failure_long"]:
        fig = plot_failure_zoom(long_results[name], name)
        if fig:
            show_fig(fig)